# Real-time Tie Detection with YOLOv8
Detects if a person is wearing a tie using pretrained YOLOv8n (no custom training)

## Environment Setup
Run the cell below once to create a virtual environment and install dependencies.

In [9]:
import subprocess
import sys
import os

# Create venv and install dependencies (run once, skip after)
venv_path = os.path.join(os.getcwd(), '.venv')
if not os.path.exists(venv_path):
    subprocess.run([sys.executable, '-m', 'venv', venv_path], check=True)
    pip_path = os.path.join(venv_path, 'Scripts', 'pip.exe')
    subprocess.run([pip_path, 'install', 'ultralytics', 'opencv-python'], check=True)
    print("✅ venv created and packages installed!")
else:
    print("✅ venv already exists")

✅ venv already exists


## Install Dependencies (Google Colab / Direct Use)
Use this cell if running on Google Colab or if not using the venv above.

In [10]:
# Use this for Google Colab or if not using venv
# !pip install ultralytics opencv-python -q

## Imports

In [3]:
import cv2
import gc
from ultralytics import YOLO

## Load YOLOv8 Model
Load the pretrained YOLOv8n (nano) model - smallest and fastest variant.

In [4]:
# Load YOLOv8 nano model (smallest, fastest, low memory)
model = YOLO("yolov8n.pt")
print("✅ Model loaded")

# COCO class indices
PERSON_CLASS = 0
TIE_CLASS = 27

✅ Model loaded


## Main Detection Function
Two-stage pipeline:
1. **Person Detection** - Detect persons in frame
2. **ROI Extraction** - Extract chest-to-waist region (rule-based)
3. **Tie Detection** - Detect ties within the ROI

In [5]:
def detect_tie_realtime(source=0):
    """
    Real-time tie detection from webcam or video.
    
    Args:
        source: 0 for webcam, or path to video file
    """
    cap = cv2.VideoCapture(source)
    
    if not cap.isOpened():
        print("❌ Cannot open video source")
        return
    
    print("🎥 Press 'q' to quit")
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        status_text = "NO PERSON"
        status_color = (0, 0, 255)  # Red
        
        # ============================================
        # STEP 1: Detect persons
        # ============================================
        results = model.predict(
            frame, 
            classes=[PERSON_CLASS],
            imgsz=320,          # Smaller size for speed
            conf=0.5,           # Confidence threshold
            max_det=10,         # Max detections
            verbose=False       # Suppress output
        )
        
        persons = results[0].boxes
        
        if len(persons) > 0:
            status_text = "NO TIE"
            status_color = (0, 0, 255)  # Red
            
            for box in persons:
                # Get person bounding box
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                height = y2 - y1
                
                # Draw person box (blue)
                cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 0), 2)
                cv2.putText(frame, "Person", (x1, y1 - 10), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)
                
                # ============================================
                # STEP 2: Extract chest-to-waist ROI (rule-based)
                # ============================================
                roi_top = int(y1 + 0.25 * height)
                roi_bottom = int(y1 + 0.60 * height)
                roi_left = x1
                roi_right = x2
                
                # Clamp ROI to frame bounds
                roi_top = max(0, roi_top)
                roi_bottom = min(frame.shape[0], roi_bottom)
                roi_left = max(0, roi_left)
                roi_right = min(frame.shape[1], roi_right)
                
                # Draw ROI box (yellow)
                cv2.rectangle(frame, (roi_left, roi_top), (roi_right, roi_bottom), 
                             (0, 255, 255), 2)
                cv2.putText(frame, "ROI", (roi_left, roi_top - 10), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 2)
                
                # Crop ROI
                roi = frame[roi_top:roi_bottom, roi_left:roi_right]
                
                if roi.size > 0:
                    # ============================================
                    # STEP 3: Detect tie in ROI
                    # ============================================
                    tie_results = model.predict(
                        roi,
                        classes=[TIE_CLASS],
                        imgsz=320,
                        conf=0.3,       # Lower confidence for tie detection
                        max_det=5,
                        verbose=False
                    )
                    
                    ties = tie_results[0].boxes
                    
                    if len(ties) > 0:
                        status_text = "TIE DETECTED"
                        status_color = (0, 255, 0)  # Green
                        
                        # Draw tie boxes on frame
                        for tie_box in ties:
                            tx1, ty1, tx2, ty2 = map(int, tie_box.xyxy[0])
                            # Offset to frame coordinates
                            tx1 += roi_left
                            tx2 += roi_left
                            ty1 += roi_top
                            ty2 += roi_top
                            cv2.rectangle(frame, (tx1, ty1), (tx2, ty2), 
                                         (0, 255, 0), 2)
                            cv2.putText(frame, "Tie", (tx1, ty1 - 5),
                                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
        
        # Display status text with background
        cv2.rectangle(frame, (10, 10), (280, 60), (0, 0, 0), -1)
        cv2.putText(frame, status_text, (20, 45), 
                   cv2.FONT_HERSHEY_SIMPLEX, 1, status_color, 2)
        
        # Show frame
        cv2.imshow("Tie Detection - Press 'q' to quit", frame)
        
        # Quit on 'q'
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
        
        # Memory cleanup
        gc.collect()
    
    cap.release()
    cv2.destroyAllWindows()
    print("✅ Detection stopped")

## Run Detection
Run with webcam (0) or provide a video file path.

In [6]:
# Run with webcam (0) or video file path
detect_tie_realtime(0)

🎥 Press 'q' to quit
✅ Detection stopped
✅ Detection stopped


## Cleanup (Optional)
Free memory when done.

In [7]:
# Free memory when done
del model
gc.collect()
print("✅ Cleanup complete")

✅ Cleanup complete


---
## Google Colab Version
Use this alternative if running on Google Colab (cv2.imshow doesn't work there).

In [8]:
# ============================================
# GOOGLE COLAB VERSION - Process uploaded video
# ============================================
# Uncomment and run this cell on Google Colab

"""
from google.colab import files
from IPython.display import display, Image, clear_output
import cv2
import gc
from ultralytics import YOLO

# Upload a video file
print("📁 Upload a video file:")
uploaded = files.upload()
video_path = list(uploaded.keys())[0]

# Load model
model = YOLO("yolov8n.pt")
PERSON_CLASS = 0
TIE_CLASS = 27

cap = cv2.VideoCapture(video_path)
frame_count = 0
max_frames = 200  # Process first 200 frames

print("🎥 Processing video...")

while cap.isOpened() and frame_count < max_frames:
    ret, frame = cap.read()
    if not ret:
        break
    
    status_text = "NO PERSON"
    status_color = (0, 0, 255)
    
    # STEP 1: Detect persons
    results = model.predict(frame, classes=[PERSON_CLASS], imgsz=320, conf=0.5, verbose=False)
    persons = results[0].boxes
    
    if len(persons) > 0:
        status_text = "NO TIE"
        
        for box in persons:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            height = y2 - y1
            
            cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 0), 2)
            
            # STEP 2: ROI extraction
            roi_top = int(y1 + 0.25 * height)
            roi_bottom = int(y1 + 0.60 * height)
            
            roi_top = max(0, roi_top)
            roi_bottom = min(frame.shape[0], roi_bottom)
            
            cv2.rectangle(frame, (x1, roi_top), (x2, roi_bottom), (0, 255, 255), 2)
            
            roi = frame[roi_top:roi_bottom, x1:x2]
            
            if roi.size > 0:
                # STEP 3: Detect tie
                tie_results = model.predict(roi, classes=[TIE_CLASS], imgsz=320, conf=0.3, verbose=False)
                
                if len(tie_results[0].boxes) > 0:
                    status_text = "TIE DETECTED"
                    status_color = (0, 255, 0)
    
    # Draw status
    cv2.rectangle(frame, (10, 10), (280, 60), (0, 0, 0), -1)
    cv2.putText(frame, status_text, (20, 45), cv2.FONT_HERSHEY_SIMPLEX, 1, status_color, 2)
    
    frame_count += 1
    
    # Display every 5th frame
    if frame_count % 5 == 0:
        clear_output(wait=True)
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        _, buffer = cv2.imencode('.jpg', frame_rgb)
        display(Image(data=buffer.tobytes()))
        print(f"Frame {frame_count}/{max_frames} - {status_text}")

cap.release()
gc.collect()
print("✅ Processing complete!")
"""

'\nfrom google.colab import files\nfrom IPython.display import display, Image, clear_output\nimport cv2\nimport gc\nfrom ultralytics import YOLO\n\n# Upload a video file\nprint("📁 Upload a video file:")\nuploaded = files.upload()\nvideo_path = list(uploaded.keys())[0]\n\n# Load model\nmodel = YOLO("yolov8n.pt")\nPERSON_CLASS = 0\nTIE_CLASS = 27\n\ncap = cv2.VideoCapture(video_path)\nframe_count = 0\nmax_frames = 200  # Process first 200 frames\n\nprint("🎥 Processing video...")\n\nwhile cap.isOpened() and frame_count < max_frames:\n    ret, frame = cap.read()\n    if not ret:\n        break\n\n    status_text = "NO PERSON"\n    status_color = (0, 0, 255)\n\n    # STEP 1: Detect persons\n    results = model.predict(frame, classes=[PERSON_CLASS], imgsz=320, conf=0.5, verbose=False)\n    persons = results[0].boxes\n\n    if len(persons) > 0:\n        status_text = "NO TIE"\n\n        for box in persons:\n            x1, y1, x2, y2 = map(int, box.xyxy[0])\n            height = y2 - y1\n\n  